# 10-Seed Analysis Walkthrough

Replicates the `Agreed analysis` reporting framework for the multi-seed generators experiment.

**Design:** 15 datasets × 8 generators × 10 seeds = **1,200 runs**.

Primary result: **Mean ± SD** across the 10 individual seed observations (`ddof=1`), not the mean of batch means.

Run the full pipeline from the shell:

```bash
cd "/home/gopi_b/SYNTH_BENCHMARK/multi seed generators/analysis"
../../.venv/bin/python run_all.py
```

In [ ]:
from pathlib import Path
import pandas as pd

ANALYSIS = Path('.').resolve()
if ANALYSIS.name != 'analysis':
    ANALYSIS = Path('/home/gopi_b/SYNTH_BENCHMARK/multi seed generators/analysis')

import sys
sys.path.insert(0, str(ANALYSIS))

from src.validate import run_validation
from src.load_results import load_raw_long, aggregate_10_seed

print('Analysis root:', ANALYSIS)

## 1. Validate batch provenance and 1,200-run completion

In [ ]:
summary = run_validation()
summary

## 2. Seed-level long table (with batch labels)

In [ ]:
long_path = ANALYSIS / 'data' / 'seed_level_long.csv'
if long_path.exists():
    long_df = pd.read_csv(long_path)
else:
    long_df = load_raw_long()

print(long_df.columns.tolist())
print('rows', len(long_df))
print(long_df.groupby(['batch_label','seed']).size().head(20))
long_df.head()

## 3. Aggregate across 10 seeds (Mean ± SD, ddof=1)

In [ ]:
agg_path = ANALYSIS / 'data' / 'agg_10seed_mean_sd.csv'
if agg_path.exists():
    agg = pd.read_csv(agg_path)
else:
    agg = aggregate_10_seed(long_df)

print(agg[['dataset_id','generator','metric_name','mean','sd','n_seeds','mean_sd']].head(10))
print('n_seeds distribution:', agg['n_seeds'].value_counts().to_dict())

## 4. Key manuscript outputs

- `figures/main/` — curated main figures
- `figures/supplementary/` — heatmaps + seed-stability panels
- `tables/main/` — Mean±SD + Friedman/Nemenyi
- `tables/supplementary/` — all 10 seed columns
- `reports/` — validation, captions, summary

In [ ]:
from IPython.display import display, Markdown

main_dir = ANALYSIS / 'figures' / 'main'
files = sorted(main_dir.glob('Fig_*.png')) if main_dir.exists() else []
display(Markdown(f'**Main figures packed:** {len(files)}'))
for f in files[:12]:
    print(f.name)